# 清洗后客户授信额度分布分析

本 notebook 直接读取主运行文档生成的 `data_cleaned.csv`，分析原始授信额度分布。

分箱规则：10 万元以下每 1 万元一档；10 万元及以上每 10 万元一档。

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

CLEANED_FILE = Path('data_cleaned.csv')
CSV_ENCODING = 'utf-8-sig'
OUTPUT_FIGURE = Path('credit_limit_distribution_cleaned.png')

# 尽量兼容本机常见中文字体。
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('清洗后数据文件:', CLEANED_FILE.resolve())

## 1. 读取清洗后数据

In [ ]:
if not CLEANED_FILE.exists():
    raise FileNotFoundError(f'未找到清洗后数据文件：{CLEANED_FILE.resolve()}。请先运行主 notebook 的数据清洗部分。')

df_cleaned = pd.read_csv(CLEANED_FILE, encoding=CSV_ENCODING)
credit_col = 'credamt' if 'credamt' in df_cleaned.columns else '授信额度'
if credit_col not in df_cleaned.columns:
    raise KeyError('清洗后数据中未找到 credamt 或 授信额度 字段。')

credit_raw = pd.to_numeric(df_cleaned[credit_col], errors='coerce')
missing_count = int(credit_raw.isna().sum())
negative_count = int((credit_raw < 0).sum())
credit = credit_raw[credit_raw.notna() & (credit_raw >= 0)].astype(float)
if credit.empty:
    raise ValueError('授信额度字段没有可用于绘图的非负数值。')

print(f'数据规模：{len(df_cleaned):,} 行 × {len(df_cleaned.columns)} 列')
print(f'授信额度字段：{credit_col}')
print(f'有效样本：{len(credit):,}；缺失/非数值：{missing_count:,}；负值：{negative_count:,}')

## 2. 描述性统计

In [ ]:
summary = credit.describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_frame('授信额度（元）')
summary

## 3. 自定义分档直方图

- `[0, 1万)` 至 `[9万, 10万)`：每档宽 1 万元；
- `[10万, 20万)` 起：每档宽 10 万元；
- 最后一档右端点包含最大值。

In [ ]:
fine_edges = np.arange(0, 100_000 + 10_000, 10_000, dtype=float)
max_credit = float(credit.max())
coarse_top = max(200_000, math.ceil(max_credit / 100_000) * 100_000)
coarse_edges = np.arange(200_000, coarse_top + 100_000, 100_000, dtype=float)
bin_edges = np.concatenate([fine_edges, coarse_edges])
counts, _ = np.histogram(credit, bins=bin_edges)

labels = []
for i, (lo, hi) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
    right = ']' if i == len(bin_edges) - 2 else ')'
    labels.append(f'[{lo / 10_000:g}万, {hi / 10_000:g}万{right}')

distribution = pd.DataFrame({
    '额度区间': labels,
    '下界（元）': bin_edges[:-1],
    '上界（元）': bin_edges[1:],
    '客户数': counts,
    '累计占比': np.cumsum(counts) / len(credit),
})
distribution[['额度区间', '客户数', '累计占比']].style.format({'客户数': '{:,}', '累计占比': '{:.2%}'})

In [ ]:
fig_width = max(13, len(counts) * 0.72)
fig, ax = plt.subplots(figsize=(fig_width, 7))
x = np.arange(len(counts))
bars = ax.bar(x, counts, width=0.82, color='#4C78A8', edgecolor='white', linewidth=0.8)

ax.set_title('清洗后客户授信额度分布', fontsize=16, pad=14)
ax.set_xlabel('授信额度区间')
ax.set_ylabel('客户数')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=55, ha='right')
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:,.0f}'))
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

for bar, count in zip(bars, counts):
    if count > 0:
        ax.annotate(f'{count:,}',
                    (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8, rotation=90)

fig.text(0.5, 0.01, '10万元以下：1万元/档；10万元及以上：10万元/档', ha='center', color='#555555')
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig(OUTPUT_FIGURE, dpi=160, bbox_inches='tight')
plt.show()
print(f'图表已保存：{OUTPUT_FIGURE.resolve()}')